# Hugging Face দিয়ে Fine-tuning (PEFT + TRL)

এই lesson-টি বাস্তব, industry-standard fine-tuning workflow-এর একটি **REFERENCE walkthrough**: `transformers.AutoModelForCausalLM` + `peft.LoraConfig` / `get_peft_model` + `trl.SFTTrainer`। এই environment-এ `transformers`, `peft` ও `trl` install করা **নেই** (বাকি course-টি raw PyTorch-এই থাকে), তাই এই notebook:

1. তিনটি library-ই import করার চেষ্টা করে।
2. যদি সেগুলো অনুপস্থিত থাকে, একটি স্পষ্ট, কাজে লাগানোর মতো বার্তা প্রিন্ট করে এবং পরিচ্ছন্নভাবে বেরিয়ে যায় (exit code 0) — কোনো বিভ্রান্তিকর traceback ছাড়া।
3. যদি সেগুলো install করা **থাকে**, একটি ছোট প্রকৃত fine-tuning job চালায়: একটি ক্ষুদ্র causal LM load করে, সেটিকে একটি LoRA adapter দিয়ে wrap করে, এবং এই lesson-এর README-তে ব্যাখ্যা করা প্রতিটি config option ব্যবহার করে `SFTTrainer` দিয়ে একটি toy instruction dataset-এ কয়েক ধাপ train করে।

নিচের function-গুলোতে (`build_lora_config`, `build_model_and_tokenizer`, `run_sft_training`) সাম্প্রতিক transformers/peft/trl version-এর জন্য প্রকৃত, নির্ভুল API ব্যবহার আছে — library-গুলো install থাকলে এগুলো আসলেই কাজ করবে এমনভাবে লেখা, শুধু pseudocode হিসেবে নয়।

Runtime note: library-গুলো install করা **থাকলে**, `run_sft_training` একটি ছোট model/CPU-তেও কয়েক মিনিট সময় নেবে (একটি pretrained checkpoint load করা + একটি ছোট training loop)। এই environment-এ এটি কখনো সেই code path-এ পৌঁছায় না।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। library-গুলো না থাকলে শেষ cell-এর `main()` dependency-check বার্তা প্রিন্ট করে `sys.exit(0)` কল করে — Jupyter-এ এটি একটি `SystemExit` বার্তা হিসেবে দেখাবে, এটি স্বাভাবিক।

In [ ]:
import sys

REQUIRED_PACKAGES = ["transformers", "peft", "trl"]

## 0. Dependency check

`check_dependencies` প্রতিটি প্রয়োজনীয় library আলাদাভাবে import করার চেষ্টা করে এবং অনুপস্থিতগুলোর একটি list ফেরত দেয়, যাতে প্রথম ব্যর্থতাতেই থেমে না গিয়ে কী কী নেই তার পূর্ণ চিত্র পাওয়া যায়। Cell-এর শেষে এটি চালানো হয় — ফেরত দেওয়া list-টিই cell-এর output (খালি list মানে সব library আছে)।

In [ ]:
def check_dependencies():
    """প্রতিটি প্রয়োজনীয় library স্বাধীনভাবে import করার চেষ্টা করো, যাতে
    user শুধু প্রথম ব্যর্থতাটি নয়, কী কী অনুপস্থিত তার পূর্ণ চিত্র পায়।"""
    missing = []
    for pkg in REQUIRED_PACKAGES:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)
    return missing


check_dependencies()

## 1. LoRA config: Lesson 2-এর `LoRALinear`, এবার প্রকৃত `peft.LoraConfig` হিসেবে

`build_lora_config` Lesson 2-এর স্ক্র্যাচ-থেকে-তৈরি `LoRALinear`-এর প্রতিটি ধারণাকে (rank `r`, scaling `alpha/r`, কোন weight matrix-গুলো A/B update পায়) একটি production-grade `LoraConfig`-এ প্রকাশ করে। প্রতিটি field কীসের সাথে map করে তা README-এর section 2-এ আছে। এর পরের সব function library ছাড়া চালানো যায় না, তাই এগুলো এখানে কেবল সংজ্ঞায়িত করা হয়।

In [ ]:
# ---------------------------------------------------------------------------
# প্রকৃত workflow। নিচের প্রতিটি function এমনভাবে লেখা যে transformers/peft/trl
# install থাকলে এটি আসলেই চলবে — এটি সরলীকৃত pseudocode নয়।
# ---------------------------------------------------------------------------

def build_lora_config():
    """Lesson 2-এর স্ক্র্যাচ-থেকে-তৈরি LoRALinear-এর প্রতিফলন (সেখানকার README
    section 5), এবার একটি production workflow যে প্রকৃত peft.LoraConfig ব্যবহার
    করে সেভাবে প্রকাশিত। প্রতিটি field কীসের সাথে map করে তা এই lesson-এর
    README section 2-এ দেখুন।"""
    from peft import LoraConfig, TaskType

    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,   # peft-কে বলে একটি causal-LM head সঠিকভাবে wrap করতে
        r=16,                            # LoRA rank — Lesson 2 section 2-এর `r`
        lora_alpha=32,                   # LoRA scaling-এর লব — Lesson 2-এর সূত্রে alpha/r
        lora_dropout=0.05,               # শুধু LoRA path-এ প্রয়োগ করা dropout, একটি হালকা regularizer
        target_modules=["q_proj", "v_proj"],  # কোন frozen weight matrix-গুলো একটি A/B update পায়
        bias="none",                     # সব bias term frozen রাখো; শুধু A/B gradient পায়
    )

## 2. একটি প্রকৃত pretrained model ও tokenizer load করা

`build_model_and_tokenizer` একটি ছোট pretrained causal LM (`sshleifer/tiny-gpt2`) ও তার tokenizer load করে, pad token না থাকলে EOS token-কে pad হিসেবে সেট করে, তারপর `get_peft_model` দিয়ে সেটিকে সেকশন 1-এর LoRA adapter দিয়ে wrap করে এবং trainable parameter-এর সংখ্যা প্রিন্ট করে — ঠিক Lesson 2-এর parameter-count তুলনা, এবার একটি প্রকৃত model-এর জন্য।

In [ ]:
def build_model_and_tokenizer(model_name="sshleifer/tiny-gpt2"):
    """একটি ছোট pretrained causal LM load করো এবং build_lora_config()-এর LoRA
    adapter দিয়ে সেটিকে wrap করো। এখানে `tiny-gpt2` ব্যবহার করা হয়েছে কেবল
    এই কারণে যে এটি একটি demo-র জন্য দ্রুত download ও fine-tune করার মতো ছোট —
    প্রকৃত ব্যবহারের জন্য যেকোনো বাস্তব causal LM checkpoint (যেমন একটি
    Llama/Mistral/Qwen variant) বসিয়ে দিন।"""
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import get_peft_model

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(model_name)

    lora_config = build_lora_config()
    model = get_peft_model(base_model, lora_config)

    # peft-এর নিজস্ব utility — হুবহু Lesson 2-এর parameter-count তুলনা, তবে
    # Lesson 2-এর toy matrix-এর বদলে একটি প্রকৃত loaded model-এর জন্য গণনা করা।
    model.print_trainable_parameters()

    return model, tokenizer

## 3. Data: instruction ফরম্যাটিং

`build_toy_dataset` Lesson 4-এর স্ক্র্যাচ-থেকে-তৈরি example-এর মতোই Instruction/Response আকৃতির একটি ক্ষুদ্র instruction dataset বানায়, এবং প্রতিটি example-কে `### Instruction:` / `### Response:` ফরম্যাটে একটি `text` field-এ রূপান্তর করে — README section 3-এর "chat template"-এর সরলতম রূপ।

In [ ]:
def build_toy_dataset():
    """Lesson 4-এর স্ক্র্যাচ-থেকে-তৈরি example-এর মতোই Instruction/Response
    আকৃতির একটি ক্ষুদ্র instruction dataset — SFTTrainer একটি Hugging Face
    `datasets.Dataset` (অথবা dict-এর একটি list) আশা করে, যাতে এমন একটি text
    field থাকে যা এটি format করতে পারে।"""
    from datasets import Dataset

    examples = [
        {"instruction": "Uppercase the word cat.", "response": "CAT"},
        {"instruction": "Reverse the word dog.", "response": "god"},
        {"instruction": "Uppercase the word sun.", "response": "SUN"},
        {"instruction": "Reverse the word run.", "response": "nur"},
    ]

    def to_text(example):
        # এই lesson-এর README section 2 যে ধাপটিকে সরলতম রূপে "chat template"
        # বলে, সেই formatting ধাপ — role-tagged data-র জন্য SFTTrainer এখানে
        # একটি প্রকৃত tokenizer.apply_chat_template call-ও নিতে পারে।
        example["text"] = (
            f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"
        )
        return example

    return Dataset.from_list(examples).map(to_text)

## 4. `SFTTrainer` দিয়ে training এবং merge করা

`run_sft_training` প্রকৃত training call: `trl.SFTTrainer` Lesson 4-এ স্ক্র্যাচ থেকে বানানো masked-loss mechanism-কে (response-only cross-entropy) একটিমাত্র `.train()` call-এর পেছনে মুড়ে রাখে। `SFTConfig`-এ batch size, epoch, learning rate, `dataset_text_field` ও `max_seq_length` সেট করা হয়। Training শেষে `merge_and_unload()` দিয়ে LoRA adapter-টি base model-এ merge করা হয় deployment-এর জন্য — Lesson 2 section 3-এর "বিনা খরচে merge" বৈশিষ্ট্য, এবার peft-এর প্রকৃত API দিয়ে।

In [ ]:
def run_sft_training():
    """প্রকৃত training call — trl.SFTTrainer Lesson 4-এ স্ক্র্যাচ থেকে বানানো
    masked-loss mechanism-কে (response-only cross-entropy) একটিমাত্র `.train()`
    call-এর পেছনে মুড়ে রাখে, formatted example পড়ার জন্য একটি
    `dataset_text_field` দেওয়া থাকলে।"""
    from trl import SFTConfig, SFTTrainer

    model, tokenizer = build_model_and_tokenizer()
    dataset = build_toy_dataset()

    training_args = SFTConfig(
        output_dir="./sft-lora-demo",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=1,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=1,
        dataset_text_field="text",
        max_seq_length=128,
        report_to=[],  # এই demo-র জন্য wandb/tensorboard auto-logging বন্ধ রাখো
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
    trainer.train()

    # Deployment-এর জন্য LoRA adapter-টি base model-এ আবার merge করা —
    # হুবহু Lesson 2 section 3-এর "বিনা খরচে merge" বৈশিষ্ট্য, এবার স্ক্র্যাচ-থেকে-তৈরি
    # `merged_weight()` method-এর বদলে peft-এর প্রকৃত API দিয়ে।
    merged_model = trainer.model.merge_and_unload()
    return trainer, merged_model

## সবগুলো demo একসাথে

`main()` পুরো workflow একবারে চালায়: প্রথমে dependency check করে; কোনো library অনুপস্থিত থাকলে install করার নির্দেশনা প্রিন্ট করে `sys.exit(0)` দিয়ে পরিচ্ছন্নভাবে বেরিয়ে যায়; আর সব library থাকলে প্রকৃত LoRA + `SFTTrainer` demo চালিয়ে loss history ও merged model-এর type প্রিন্ট করে। আগের কোনো cell-এ training চালানো হয়নি, তাই এখানে `main()` সরাসরি চালানো হয় (কিছুই দুবার চলে না)।

In [ ]:
def main():
    missing = check_dependencies()

    if missing:
        print("=" * 78)
        print("Fine-tuning with Hugging Face (PEFT + TRL) -- dependency check")
        print("=" * 78)
        print(f"\nThis lesson's real workflow needs: {', '.join(REQUIRED_PACKAGES)}")
        print(f"Missing in this environment: {', '.join(missing)}\n")
        print("Install them with:")
        print(f"\n    pip install {' '.join(missing)}\n")
        print("(transformers/peft/trl are deliberately NOT installed in this course's")
        print("environment -- earlier lessons in this phase build every mechanism")
        print("these libraries wrap [LoRA in Lesson 2, response-masked SFT loss in")
        print("Lesson 4] from scratch in raw PyTorch, precisely so this gap doesn't")
        print("block understanding what the real library calls below are doing.)\n")
        print("This script's functions (build_lora_config, build_model_and_tokenizer,")
        print("build_toy_dataset, run_sft_training) contain real, accurate")
        print("transformers/peft/trl API usage -- read them directly, or install the")
        print("three packages above and re-run this script to execute them for real.")
        print("\nExiting cleanly (no traceback) since the libraries are unavailable.")
        sys.exit(0)

    # এই branch কেবল তখনই চলে যখন transformers/peft/trl সত্যিই install করা থাকে।
    print("All required libraries found -- running the real LoRA + SFTTrainer demo.")
    trainer, merged_model = run_sft_training()
    print("\nTraining complete. Final loss history:")
    for entry in trainer.state.log_history:
        if "loss" in entry:
            print(f"  step {entry.get('step')}: loss = {entry['loss']:.4f}")
    print(f"\nMerged model ready for deployment: {type(merged_model).__name__}")


main()  # dependency check, এবং library থাকলে প্রকৃত training